In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.9 MB/s eta 0:00:00


In [3]:
from ultralytics.utils.benchmarks import benchmark

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.5/112.6 GB disk)


In [5]:
import shutil
from pathlib import Path
import os
import yaml

# Przygotowanie danych

In [6]:
zip_path = "/content/drive/MyDrive/dane_detekcja_pojazdow/EAGLE_dataset_1024_res_cleaned_obb.zip" #mniejszy zbiór

!cp "{zip_path}" /content/

In [7]:
!unzip -q /content/EAGLE_dataset_1024_res_cleaned_obb.zip -d /content/data

In [8]:
data_path = Path("/content/data/content/data_1024_res")
yaml_file_path = data_path / "data.yaml"

In [9]:
data_content = {
    'path': '../EagleDatasetYOLO',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['vehicle']
}

with open(yaml_file_path, 'w') as f:
    yaml.dump(data_content, f, default_flow_style=False)

print(f"YAML file created at: {yaml_file_path}")
print("Content:")
print(yaml.dump(data_content, indent=2))

YAML file created at: /content/data/content/data_1024_res/data.yaml
Content:
names:
- vehicle
nc: 1
path: ../EagleDatasetYOLO
test: images/test
train: images/train
val: images/val



In [10]:
with open(yaml_file_path, 'r') as f:
    yaml_content = yaml.safe_load(f)

print(yaml.dump(yaml_content, indent=2))

names:
- vehicle
nc: 1
path: ../EagleDatasetYOLO
test: images/test
train: images/train
val: images/val



In [11]:
# Assuming data_path and yaml_file_path are already defined
# (based on previous notebook state)

# Ensure data_path is a string for the YAML content
updated_data_path_str = str(data_path)

# Update the 'path' attribute in the loaded YAML content
yaml_content['path'] = updated_data_path_str

# Save the updated YAML content back to the file
with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f)

print(f"'path' attribute in {yaml_file_path} updated to: {yaml_content['path']}")

# Display the updated YAML content to verify
print("\nUpdated YAML content:")
print(yaml.dump(yaml_content, indent=2))

'path' attribute in /content/data/content/data_1024_res/data.yaml updated to: /content/data/content/data_1024_res

Updated YAML content:
names:
- vehicle
nc: 1
path: /content/data/content/data_1024_res
test: images/test
train: images/train
val: images/val



# Wykonanie benchmarków

In [12]:
# wykonanie benchamrku
model_path = '/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt'

## TensorRT



### half=false

In [13]:
benchmark(
    model = model_path,
    data = yaml_file_path,
    imgsz=1024,
    device=0,
    half=False,
    format='engine'
)


Setup complete ✅ (2 CPUs, 12.7 GB RAM, 55.0/112.6 GB disk)

Benchmarks complete for /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt on /content/data/content/data_1024_res/data.yaml at imgsz=1024 (297.15s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+---------------------------------------------------------------------------------------------+
|     Format     Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS    |
+=============================================================================================+
| 1   TensorRT   ✅         14.8        0.7754                8.17                     122.43 |
+---------------------------------------------------------------------------------------------+



,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""TensorRT""","""✅""","""14.8""","""0.7754""","""8.17""","""122.43"""


### half=true

In [14]:
benchmark(
    model = model_path,
    data = yaml_file_path,
    imgsz=1024,
    device=0,
    half=True,
    format='engine'
)

Setup complete ✅ (2 CPUs, 12.7 GB RAM, 55.1/112.6 GB disk)

Benchmarks complete for /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt on /content/data/content/data_1024_res/data.yaml at imgsz=1024 (419.14s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+--------------------------------------------------------------------------------------------+
|     Format     Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS   |
+============================================================================================+
| 1   TensorRT   ✅         7.5         0.7746                6.79                     147.3 |
+--------------------------------------------------------------------------------------------+



,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""TensorRT""","""✅""","""7.5""","""0.7746""","""6.79""","""147.3"""


### int8

In [17]:
benchmark(
    model = model_path,
    data = yaml_file_path,
    imgsz=1024,
    device=0,
    int8=True,
    format='engine'
)

Setup complete ✅ (2 CPUs, 12.7 GB RAM, 55.2/112.6 GB disk)

Benchmarks complete for /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt on /content/data/content/data_1024_res/data.yaml at imgsz=1024 (1001.77s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+---------------------------------------------------------------------------------------------+
|     Format     Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS    |
+=============================================================================================+
| 1   TensorRT   ✅         6.0         0.7508                5.88                     170.18 |
+---------------------------------------------------------------------------------------------+



,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""TensorRT""","""✅""","""6.0""","""0.7508""","""5.88""","""170.18"""


## Zbiorczy benchmark

In [18]:
benchmark(
    model=model_path,
    data=yaml_file_path,
    imgsz=1024,
    half=True,           # Zostawiamy dla porównania
    device=0,
)

Setup complete ✅ (2 CPUs, 12.7 GB RAM, 55.6/112.6 GB disk)

Benchmarks complete for /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt on /content/data/content/data_1024_res/data.yaml at imgsz=1024 (2171.18s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+-----------------------------------------------------------------------------------------------------------+
|      Format                  Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS    |
+===========================================================================================================+
| 1    PyTorch                 ✅         5.7         0.7871                25.19                    39.69  |
| 2    TorchScript             ✅         5.7         0.7746                12.82                    78.01  |
| 3    ONNX                    ✅         5.3         0.7746                18.05                    55.39  |
| 4    OpenVINO           

,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""PyTorch""","""✅""","""5.7""","""0.7871""","""25.19""","""39.69"""
"""2""","""TorchScript""","""✅""","""5.7""","""0.7746""","""12.82""","""78.01"""
"""3""","""ONNX""","""✅""","""5.3""","""0.7746""","""18.05""","""55.39"""
"""4""","""OpenVINO""","""❌""","""0.0""","""-""","""-""","""-"""
"""5""","""TensorRT""","""✅""","""7.4""","""0.7746""","""7.02""","""142.41"""
"""6""","""CoreML""","""❌""","""0.0""","""-""","""-""","""-"""
"""7""","""TensorFlow SavedModel""","""❌""","""0.0""","""-""","""-""","""-"""
"""8""","""TensorFlow GraphDef""","""❌""","""0.0""","""-""","""-""","""-"""
"""9""","""TensorFlow Lite""","""❌""","""0.0""","""-""","""-""","""-"""
"""10""","""TensorFlow Edge TPU""","""❌""","""0.0""","""-""","""-""","""-"""


# Eksport do wybranego formatu

In [14]:
from ultralytics import YOLO

model = YOLO(model_path)

model.export(
    format='engine',
    imgsz=1024,
    half=True,
    device=0,
    workspace=4,
    simplify=True,
    data=yaml_file_path
)

Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n-obb summary (fused): 109 layers, 2,653,918 parameters, 0 gradients, 6.6 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 6, 21504) (5.7 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 14 packages in 421ms
Prepared 6 packages in 6.03s
Installed 6 packages in 386ms
 + colorama==0.4.6
 + coloredlogs==15.0.1
 + humanfriendly==10.0
 + onnx==1.20.0
 + onnxruntime-gpu==1.23.2
 + onnxslim==0.1.82

requirements: AutoUpdate success ✅ 7.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to 

'/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine'